# MPNN → ColabFold → Amber relax → HADDOCK3 emscoring → PRODIGY

上传 PDB，修改下面两个参数 cell，然后依次运行全部 cell。最终下载带 ipTM、pTM、HADDOCK3 与 PRODIGY 分数的 CSV，以及 HADDOCK3 最小化结构 ZIP。


In [ ]:
# 0. 上传一个多链 PDB并识别链顺序
from google.colab import files
from pathlib import Path
from collections import OrderedDict
import csv,hashlib,importlib.util,json,math,os,re,shutil,subprocess,sys,traceback
u=files.upload(); p=[(n,b) for n,b in u.items() if n.lower().endswith('.pdb')]
if len(p)!=1: raise ValueError('请一次只上传一个 .pdb 文件。')
ROOT=Path('/content/LigandMPNN'); INPUT_DIR=Path('/content/user_inputs'); INPUT_DIR.mkdir(parents=True,exist_ok=True)
USER_PDB=INPUT_DIR/Path(p[0][0]).name; USER_PDB.write_bytes(p[0][1])
AA3=dict(zip('ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL MSE'.split(),'ARNDCQEGHILKMFPSTW YVM'.replace(' ','')))
r=OrderedDict()
for l in USER_PDB.read_text(errors='replace').splitlines():
 if l[:6].strip() in {'ATOM','HETATM'} and l[12:16].strip()=='CA' and l[16:17] in {' ','A'} and l[17:20].strip().upper() in AA3:
  c=l[21:22].strip() or '_'; k=(l[22:26].strip(),l[26:27].strip()); r.setdefault(c,OrderedDict()).setdefault(k,AA3[l[17:20].strip().upper()])
PDB_CHAIN_SEQUENCES=OrderedDict((c,''.join(x.values())) for c,x in r.items() if x); PDB_CHAIN_ORDER=list(PDB_CHAIN_SEQUENCES)
if not PDB_CHAIN_ORDER: raise ValueError('PDB 中没有识别到标准蛋白链。')
print('PDB:',USER_PDB); [print(f'{i}. chain {c}: {len(PDB_CHAIN_SEQUENCES[c])} aa') for i,c in enumerate(PDB_CHAIN_ORDER,1)]


In [ ]:
# 1. 所有用户参数：checkpoint 支持 "1,2,3"；temperature 支持 "0.1,0.01"
CHECKPOINT_OPTIONS={
1:('protein_mpnn','proteinmpnn_v_48_002.pt',''),2:('protein_mpnn','proteinmpnn_v_48_010.pt',''),3:('protein_mpnn','proteinmpnn_v_48_020.pt',''),4:('protein_mpnn','proteinmpnn_v_48_030.pt',''),
5:('ligand_mpnn','ligandmpnn_v_32_005_25.pt',''),6:('ligand_mpnn','ligandmpnn_v_32_010_25.pt',''),7:('ligand_mpnn','ligandmpnn_v_32_020_25.pt',''),8:('ligand_mpnn','ligandmpnn_v_32_030_25.pt',''),
9:('per_residue_label_membrane_mpnn','per_residue_label_membrane_mpnn_v_48_020.pt',''),10:('global_label_membrane_mpnn','global_label_membrane_mpnn_v_48_020.pt',''),
11:('soluble_mpnn','solublempnn_v_48_002.pt',''),12:('soluble_mpnn','solublempnn_v_48_010.pt',''),13:('soluble_mpnn','solublempnn_v_48_020.pt',''),14:('soluble_mpnn','solublempnn_v_48_030.pt',''),
15:('sidechain_packer','ligandmpnn_sc_v_32_002_16.pt','')}
for i,(m,f,desc) in CHECKPOINT_OPTIONS.items(): print(f'{i:>2}: {f}'+('  [仅侧链打包]' if i==15 else ''))

# LigandMPNN
TASK_CHECKPOINT_IDS='1,2,3'
TEMPERATURES='0.1,0.01'
CHAINS_TO_DESIGN='A'
SEED=112
SEQUENCES_PER_COMBINATION=100
BATCH_SIZE=10
PARSE_ATOMS_WITH_ZERO_OCCUPANCY=1
SAVE_STATS=1
FIXED_RESIDUES=''
REDESIGNED_RESIDUES=''
VERBOSE=1
PACK_SIDE_CHAINS=False
NUMBER_OF_PACKS_PER_DESIGN=1

# 每个 FASTA 独立提取 Top N，再合并去重
EXTRACT_SOURCE_GLOB='*.fa'
EXTRACT_TOP_N_PER_FASTA=20
EXTRACT_PER_FASTA_DIR_NAME='extract_per_fasta'
EXTRACT_MERGED_FASTA_NAME='merged_top_unique_sequences.fa'
EXTRACT_TSV_NAME='merged_top_unique_sequences.tsv'

# ColabFold：每条候选运行模型1–3，按ipTM选第一名，只relax该模型
RUN_COLABFOLD=True
COLABFOLD_MSA_SERVER='https://api.colabfold.com'
COLABFOLD_USE_ENV=True
COLABFOLD_USE_FILTER=True
COLABFOLD_MODEL_TYPE='alphafold2_multimer_v3'
COLABFOLD_NUM_RECYCLES=3
COLABFOLD_NUM_MODELS=3
COLABFOLD_MODEL_ORDER=[1,2,3]
COLABFOLD_NUM_SEEDS=1
COLABFOLD_USE_DROPOUT=False
COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE='auto'
COLABFOLD_MAX_MSA='auto'
COLABFOLD_CALC_EXTRA_PTM=True
COLABFOLD_MAX_BINDERS=None
COLABFOLD_JOB_PREFIX='binder_complex'
COLABFOLD_RANK_BY='iptm'
COLABFOLD_NUM_RELAX=1
COLABFOLD_RELAX_MAX_ITERATIONS=200
COLABFOLD_RELAX_MAX_OUTER_ITERATIONS=3
COLABFOLD_RELAX_TOLERANCE=2.39
COLABFOLD_RELAX_STIFFNESS=10.0
COLABFOLD_USE_GPU_RELAX=False

# HADDOCK3 emscoring + PRODIGY
RUN_HADDOCK3=True
HADDOCK3_EMSCORING_NEMSTEPS=50
RUN_PRODIGY=True
PRODIGY_TEMPERATURE_C=25.0
POSTPROCESS_CONTINUE_ON_ERROR=True

# 输出
FINAL_CSV_NAME='mpnn_colab_haddock_prodigy_scores.csv'
FINAL_STRUCTURES_DIR_NAME='haddock3_emscoring_structures'
FINAL_ZIP_NAME='mpnn_colab_haddock_prodigy_results'
KEEP_AMBER_RELAXED_TOP_MODELS=False
DELETE_COLABFOLD_INTERMEDIATES=True
DOWNLOAD_FINAL_CSV=True
DOWNLOAD_FINAL_ZIP=True


In [ ]:
# 2. 指定 de novo 链；留空表示所有链均做 MSA
COLABFOLD_DE_NOVO_CHAIN='A'
COLABFOLD_DE_NOVO_CHAIN=COLABFOLD_DE_NOVO_CHAIN.strip()
if ',' in COLABFOLD_DE_NOVO_CHAIN or (COLABFOLD_DE_NOVO_CHAIN and COLABFOLD_DE_NOVO_CHAIN not in PDB_CHAIN_ORDER): raise ValueError(f'只能指定一条现有链；可用链：{PDB_CHAIN_ORDER}')
print('de novo chain:',COLABFOLD_DE_NOVO_CHAIN or 'None')


In [ ]:
# 3. 执行完整流程（所有实现已嵌入本 Notebook）
import base64,zlib
exec(zlib.decompress(base64.b64decode('eNrdPf1z48Z1v+uvQC4zAXHHD+mcZFy1tKuTqDvFEqmQ1NkXHoMBCVCCBQI0AJ5OVjRzaZ2xPT3blw87rWM748nHZNJ8OG0mTe1r/L80pu7uJ/8LfW93AewuFhTPcdtpPXMmCbx9+/bt27fva1df1J6oatvuvuXbO7vNpjb7zasPf/KdP9/59oOPXprde+Xhx/cevn8Xfj66c0/bXOt017RuMNGa8ATezv79D5/88fXZnX96dOfVs5/cOfvxz5aWbGekuX4UW55negTveOL7JWN1SYP/9r1gYHnaTmujsW1ubLXJQ3ektVutbtW57UZxlIDif9HBNHa9ajiOQ8cpIZBB3kXTwSQMhk4UVcOpX0rhe/q+G+tlTR96ge/gl0rFdibxAX5dwf8dxPEkWq3VAO5gOqgOg3HNtqYTK7SiWsaGKkUTxSHttF9OuxgeOMPDejecOvSZkqBeNoLjCMblDKexNfAcJGiMZEzcCX4wRhFCX6DkTif7oWUD7SkKfTcMNo7r9cvVr1bJGAZuMDmODwL/qfpK9UnyaOxVhoHnOcPYDfyoXl+urlBYe1xB3tEnTzKs/TI3DGOJPEunRKuT2dBqgDWwHc9E5owjXYSqjg9tNyzBO8ePI8oPjUygGSR4VYzRB1Z0wLEWu9l3YpPvqgoQBgVJ+zMkmhG3HRz5XgDMsoHmk4kVH1R9a+xooyDU8BfIIUcwil5Jv1idxLpxSto7tyfAMNrajZ1xb6VP2uJ3bLt+rbH+zG5rq9k1W7vdrVazU71leVMHRJQiGLtR5Pr70D4KQkBUSjFWOOKMRMgZeCbeoeVGjtae+rE7dhphGIQl/cH9D2cffPfs3b9/9PJrn95/W9cuaTiP1ecD1y8xDAZFef44s66ge8IgkLe4ZMCHGbkvOtrfaCvLl7+sXbyoXc5gC0gb6ZSqs7de/uSjPzx467uf3L//yR9fm/3m7tmbvwdSyQyc6kyeApB7/5YbBn5P77ba69fMTfh/w2y2zGcbW1evdTtmq7l9w9xurW3ofWAhrE/SEMTEnBxzUggPqpNj+jIKpuHQgZcUqho6lm3Gzu245PjDwAbe1PVpPKo8qRsiPP0C8BPPGjqZxtDjIBweVHGmSkS+JsDn2MShlLWxNTG9YGjhoqrbzi136Bj8wnzMtmXtyHH3D+LIDHzvuL5peVGK7zOQa4X7UVXq14yGj0X24+A4n3w2KUchrCA6K3QcoBlys0Ma+NPx5Ni0PNeKnAiXYbY09JsDf3KzCkTdHOiroClRISdPR0A8fU6+cW8GQeCRF/iFG3TyPhg8DyuUQNCvXFtQOOQFfHJPYYuYeM5t8oZ9Z3hPc6uQbGNhugKP+RUYeKhoyCI8T2oJa5wjAIdG6RPWUeyEPlBHJWMM2hc7FhhZRQUm7KMZwtCpgkouqfCUEcTgNQY2+QIhQkRFBsHNMsCpppiAhjBzJZ2zMHDwx6vAYLqdU4sBFH/kcPaCObJAUyGZB2wYoTMMQhuFpNcnDw4AjxPC7yZs9XT1OC9MgQoHt5GYg0TGhdYRoPcdZNYuYKWo581ENZp4boxtBFYSJPUUHwqNOymJbAtiAiYybRiALvWnDg/JMCC5R2CQlPSndGnWAIgN1I0IYhytCMJxp2pNJo5vl0q0DewbbOMQWWMYhoAhZSXS01tZ7Qtvi9hKtlDQAatzoBOCEHG6D84b0GcbSOjE09BPGjOZglkNrWFsxsHEnPoutDNHYTDmRKus4TufsZzIoC2KTdJ90i2Kz/mySmwDKx4esAXnWKBwS6Feenr1m98q34wuGsEtJ0QLGYRi5NqIuV7qVS4tV/6q6jT6l0Ctsq5zgkXwniNZcXicg2DdAEVEZZYInup+GEwnpRVOHpzbQzCYteto6JB9/zwpts35Q3VtGNo3y+pB0RkzXWR7giklKllbOPS0G5Q4hoau0JJe1o3ecj+3Eul8JpJ0IgxDz0+AvsqxqSxCp2QCUPpdgklEBECSrwlJ1SnQEJaMrMWpwYlcFa3H0qFzXPes8cC2iAG6Sv7fUxHaR7UNjyOHN7QdLzFmmfxGjuOjKeHEpcxeTGxb2rNgH9L+0mH0EQxxnCMACFK1bLsktzc4EEpbMhcIKQi25+DKpkCG9lSdrkux4wHo6kN+tScN2HIHnMBkC17BrmTtl4ipnm4epMVIPyFPV6sry/unempe6RXiQ+lG9qRKvDQ92Z/QsuFWPIhi6N6WnNq9TqNttva6ZW2n0b7a2DAbz3Xba+tds9u5nntGfGm+9fpac2NrY63b6JQ1pq38wDdt/LgVUHMp6YEzjoNpPJnGkQ7fydvdjSsgdc44scciE9w0gE+bQhtuIAhBLWvgHVVk+QYJzbvwhNCNfobZXNtpLOWd9IwL7r4fwFw4qEQiTlITqhZ1IQXKFmm0lPrp1Kgdeda+aF3q4JPGjgsTCUwAu65S4W1g/h1nNlO2KZvwr7gWUeBNwetXNhHecW1wtKETuTYIgGcNwCkeO+NBaPlqNOeBc5ipoC2CdA4kb/ky/47YoHXO4QOTeXLMdA4H09N3b3SvtZp7zSt7m5uNdkP0+wTIhT1G5siTnd6xzZyVyIIP8IqIkGQVgj0/cH3i5Jiubzu3yxrvyYGO59WKgRrRAVMbfzul9l7TXG/tXNlqrpHgQFlb4fd/EtCIjydOmZdFDFCUNRPIyEcXekLXmZGFyocENmDv1gEgNk8EyNXly/apeULMWqkrgyiDU7Nrnsj6kR/YqS50RpUK9bKpAqmlRGSBsGA8BqFHhooGYC7iNUVdyrx4adOsVDI+ARTHNHHTkRZzLwPsS4EiIFXmQa5P2LJsFoPqNBobeYCJPSCuMANKNGseEDhljsA/ckIGmrEvDzxAA4bEXRjwlbUuSHln6xuNPDCI2QDWdjAySTMnYm2aeztXUMNumqR1o5Nvys0sa8TPdX6wxJ614mAcmeiFmC86YWAGw+F0YvnDY4Zid63daZhr3dZOx3x2q3vN/Eaj3TJb6+t7u2vN9RsKJlu3HBPDTQnlnbXrDRN2j66CZLBmBkGUkHu90b7S6vBM6fPWwvq1ta1mx+y2zI1GZ+tqMzGzZEOFyuclEFCi3iwXhDkOYEuNYGuCrorwCJ1tbj0H23YbADb2Gp2Fuhq5t4kqIloZR6/GIXQD2pDQ8Lh9hQ4djtjhHGxCr7tr68+A/G00TMqL4q5yHicRnOEhiDOYKJS5LMiuAs2HmPT8ulVoxJWv9HsrfUOJM1shSEhEtsJ0bsWlguPsEPuF8kVC2F/KnAYar7jpk8BrXdcuaivLy4b0fqSvZzuHdpLbRk5rJ2jSyjuEcarnMWWsqYtq/fSv+c2nzivwPB49CRGzSTNyAPmxgMBMPVTzRekUTghgEzuy62h1YqTnVp3brXGPvB1zKZF0J4htUIV1Dvvu1m4D5wb0Zcg/73Q30GRMW8vEU0qrFKHgObA31MAfwq5wbhg7J0ojPjpFE16zn/7u4e9/9un9t4snpzxncoQujEy4qBU7cj0S7GQ5A27LQOMc/KcIHBESQ0wM705rDw2hq9utK0YuIMAhVY19E140g3gzmPp2Esc/STbzU+3snV+evfqn2Ssf8ElAYuT/+c7fcVKG1hLtKYl1FvbLeaLzQzAZvnLqY3Rbu2Yz8zTEOFXmCcTRLUCfn0vRi6kp5jobu2meZBRQO6kKhJ4UkAIvo1vy1PK/cO8USawG4O6W9COdhFYxBlbXdUWkVAO79ACY7ykieyTMig7ZEPDRHyUKW9bABnLH+KSu34x1Iz9amDQagTfBgj8sp95/ZsQmkyUar3kCWNfBUamnhMqyF7SrQqA5UY1z2mRBmHNBsyCEGrJvKB/n/Ah18EjYhwStgDEkwX1YqB1Ko9iSeAnFbXkDb5XXQ3PaJJMz9TGGxdbA+fBElqEFt1IWa4hSgKGwRWRCiK89xmSro3ifRcL0OIjB4+VieIsJ0qmkoCRvM5GfjHsyPLUBTiT2nq5msqidCNGxNMHK1H9OZuenl89evXP2zquslGP27suo/9+4N3vjLdgCaCHH7MM3Zq/8MNP++UXxeYQrQf3sq/x1jCaa4mREhUHMOWNP45nSxKZRTbmTBaKcchMu6in3kk2zONB8BBT9mdb22pXN1vaGubP2nAnW4kaj3VGnR3JsEx/0VlGelPiY4Z9kEYRm80UmK/T5zi9mv7sze/2jRy+/Nrv3+tm/vA+ilBcaWiiSC4GqY4rwIoskgqHpjpwIXGrX8ahU5MJXRK+UlUu/LGvjcl7N8mlwwU3mFWRZUn9lUauVeYUl5NUFIWC1NkupbZDnyV9iIAiGwYY7jJ8VjQPCQxxzVJf4Wmw48Hs9zbGURBOQmwSVNSGKVd6mAPsB47CgO6i+qO47RJMYBDl8QXQStacyhp4gChgI5H4vzTNd4J9aPKk6VAoog6Rx71RSVfNJQLgZXWgWPwemUpR0lArv5sJTiLh+wvVyWhbNlDotf5JWTx/ALijwCX4PU3/cUipqpkh5nhTvGac3fRWSE7W6zUGzaU5jIzRAIeQAiPwlGUax7EsOC6VpxmRbEXKNpxqa2SDIuxtXaBzFbLVB5ab6XaaiooSWlXCWgS3pMkna7O53Zvf+Gauvfv2Ps3d+8eDtlx59/0/Znn3L8lzbtMauH5jW0LWTLVRfW99obF69tvXM9k5z9+vtTvf6szdYmywBJVdOPL5YppUC+EXOgrJClpAUrqjnM2H5qi7Gq9ASool/rEvBX8VcVHJypNPFQFaBBjw7e/MD2Nk0QKMBNz/5452HL/9e9HzJpAnGSCsExejYqHNLL8LIJBrKGqtNEBY5QSMWEkiY1WU7aXaUQikG3ml8fa/RXG9g/gAQ9hV+3Dl8IO20E/IBXHnz49mHPy/gBiMJhSkjqZIXuL+QBhDuR+++d/avb85+/cMHv/q5kgbaAq0Z+JDXGMYtU14Dy4rZtQClStNfJH/2ow9nv3lbGkVOsj69/6OHv/03WioNwvfo/Q/BcMLB5XrIRiuuDUx7rbJQHxWzFPD5YJCkiUSKwb/ITMKvta6Yu+3G5tZzpyahfXX5CfvU1OUWB1Z04LmDanRgrZREGqpkZ3NKhlE9cG7b7j5s1iWjt/pknwt9ZXRliqWgFgMIZy5wMgQpP5BzzsQHErS0rIhnLTyR4FmKncefn3F5raLhkjF1A7OU11tUwvKxBcHGF2BpOUtawZYyr7z0+fm5uRCFcrOXWS6EGBS7vFDLQr7nqxYkXZk5cHSZYCmeqDoF/40C1etFvDvHY6Ol2hIJQpPSELSEa8Nm1ssJTZ9phzKZHLpnpeDESkilujBqPdIFdbBKPXlKl3HK+JVqqRJsaiF8d0MsKd/prEnRV37noDhEBuTZXwUtbTsja+plypqNZ2lJSHZksd8a86m1aDoeW+GxWMbJJ8wjrOFUZjnEJtS2pkHvND6StJUDJlLblOORNnCACU5SLWM79nSSIMmFASQsjNEcMuTneuBZg02sbWVoshlVt4cp0eAzdJ106HmeS02pkZS6NNgu7wUqmxC2KeBZWHxJdcZmqaBMaWlp6Yval6vZkD+9//bZL96fvfcP2sp/3vn+E9rZ3Vc1d9Ld0c5e/97s3muf3r87e+OX2hrm1yqh41m3tbPfvnH27vvU2Qdr88GvfgU7GoDS8qiR62MxMK0s5AuvhEpEmrBLKxApHJeNIjVaDAiPS2A0zI0o7hJ9bkgKk1UrlrrHE7pXl7l9mzepKXLSTjyjhHU7yLYh8gZT+VJVV6Z82q1nO2Xu97Ot9jP8GSa0ROhaYBDnhVVef/DRr7VraxsbrfVnnqjttlsbW1dvaI/eufPw599+8NH3z957CSYC7IVPPn737O63RdwklMbFWz6/A0h+UDmyQr+CG4oHOjPiQxzo3Y4pAnuAee6Qf5vysGd5kwMLv1XAJJxGleet233tb7V9N76kOIEVBYfh8SQO/FoqoeqTSnNHmhwt0ivekKcqHGuVkVabRmENTzR4NTBravQA1RPVizXbjWKSyrbAjqnFjh8FIQjoUQ1UiVM7dEIfRK6m83Vgg3hEn5tRMAg8M5hUo2DRHnQ+ApV2Bu6Ow5rULtYuIr7LT9Vs51bNn3qe9q1vwVKaOnrh8a1o4gyxhnY8CcIY7TZSkwdrxzbxVSmbmyqpJ9FT75S0JOYyfKkGobvvcpWXtGaF5PzqtHCeAzMksPT4StZqobMO9GiEHiHHo94hhnT8SdUKMWdZQjnsVVb68LysXTaIl1QydN5SwPawK/JESLWjGT3csYWcncYjSEtBAXlZpI2qL44uQ1dUKMjDLasCFJgN1bKpSb+xiSSJmvEYc8KXVfCuj1ljBjuOLCxvAW9bBUrGxqFVwSRn1hKw5LeZLmd2Qk/VGAUuSlqC5THFA0T7ICX7dKh5xSmVpaaozKMgPKQTjGOi9aiK5nhCMbJY+SpYeC45/TgXPgNjZa8WqSHJ18jS52TvA7sC89ek5Iy1Ekte810VF7+KLdPeixswDixcLCvwYdFWKR0Ll9cyy4cPi3gApbKI0mEMjnk39uQ00UAyrmztUt8SgAVvFKtzqAcstzSKXNOVy1kcCUiJ2AlCtrRKKkO6wF3EwqJEKmvgKDPoE9rXqawJphHMqX+rnskISJnZaF7Pg4GJjBF5EXJza7vbaOeB0Rfz0Jal5/DyABOLOBKq1wcBTOc09LiuwLY1QfqvK7oKTdi2/Liuo3tQSTfoCjNcKsxuqa1Ul3VVBQ8LYOWmCj27FbLxuBExQ9AHwrkhxVtSEIvNWQ8/cxFBfJjGxXL9nFsQpKNh//DjH8x+9N7Zmx88evkNTHX94Ldg3z569z3wNDTiytIMVwKgjI3lZdxOQoQ5mSIjMthiYl6FVJus9ja5w/Hy2pvnzqYnlzNypz7KCGg3ICVXxTvSn1pZXrnpnyQ4TmHRqSNwxR66AE8Md4lFveSLeOKsKF46b3wseKqoGEXuHpMzqzhK5cByDnaKOUNjPTHGfGu6ycpKI2Nmnf9Rlk5OpkAZWSIIfY56iZVM1Qs0EQc4tEIwNSwwIo/reIr9ohAzjgzVumRCl56047ibBOT6Gf9BYnvAgSQewgQX/O3jIa1my9iBAGLIq7m3Y7Yb6zfWtzG5AJaeNY0DXThEqImJa76FwZ8wph3CFHhOaNFFNq9nhsNsrLW3b5gdLOnqtrYb7bXmeqOIEmreLYaDp21s3WZLKU8HJuJRzaRd0q5yADRrZaWncIjFrdcwuAU6uJaZSOxSBkNowHZv1TUMvD1WUttGmSkHcLp4xULOAuSLDEgJb/cGlnYmlCTywVeTMoGrs8+yVIeKzeqi+ZKBzN3yhtMoDsbpe2Ln13EOynzogRpzdVnGtteeKwsaHO28uk5xchtaduKgrhy60BW1FaW+CHBHpokuoHryheeKhwIF/zB7TUNudV4qt1Fk4N8WGAjsBIrUOF0luXapAMtNotgdjXxwrnNNOt2tzc1mo9NRkRhMgcbzCAXTGgxsNbl0WTtW6IFKQ3M7oz235kUW4mEOmdV4qqMjSo8dBhOsRhbNq412a1eoPEbQ/ck0JywIfHV3LycwRCoCjDDnxIImAzNjBdQ0vT5AKpVO1kw9+ZK9OnSciUmWMyxbk66TnPhjEgc2VZ7la81nzCs3ykv8tsMEO9mbdPEt4odNDZbP/nFd3wfnxOYP65ApsWKTuMH1leXl6jLXPlu0Q8vzBtbwUFp+I4eeO0KwIhh74iLi7AHYTUVDJmdLAI38nKngOvsUp3XowZoGKZ2EAUaj6xivEzdxfQydgV0Y6sTmUixzYq8mip6r0FpSZG4IkekClyj9zIY1cG9okoi3OYnH3KSvr22v0yixudvd4cg4BD7S+vJ6T594ATmQo08sx3w+CkixE3rR/eQiD8lZx6jnYxqmKJArVFS4Onep8FFwU0m1++jCicoGOWWYSOWKuby8Yl6sIuEXDIVRQ7tmbvvn0XcSAeA6n9gDZd8sBMyPfrGCfEWdS1aWr6ZLe/D2S6QncKFIV9rXOq0muCMXCgrUBeLYoP57qeOyBzAVjNbdjStziCQjMVENwtThHJNLaqISz1I86T8npsiNGJYHBuv41ETWAUng6gjDt1mgidSCRk7SA5NZOKmmFcpUPraEdBC5km8+w5O0l0vCrAEDykK3XKZdWLXqbPvFiylRUsqXMGOV8E16Q1/kn+PqMoljydqR+ijymalGaunmG4tcgMEDDn5Y+QSzNDxShsxtlVI9sii4rJw7I7jP01g+F3Z0DgbeddArrj/iA0EGb+lkldD81Uks+5fuAhq1FeIsVSqO3sDa1mQiS5Eh5mlb6OyyJTc4pqm9IyuS1iQqc8caHmQavapzmUVFZowkEr9STZNWGvjcsD4wYX32zp3ZB2/M7r6lXUpeM810SWM72af3f/TgBz8+e+UewD746BVtvXNdm33vLk11ad/Y2uXvAjqwbDsYHj7BpKHEzn5RnUUXZf7eEe66p2QXpTTcLKWU3jRuRhfr8E++cIX2wDt0Thy6Q+nmgJSuFCFVTLmqEYUWke5cIdkXbiT5WhCDO22PAgHv/DiaP/Bb9pF4l0wZjw5Ij7jz3rrtRIF3SwIAk1B6MgC7SmLZknjSj5dnjA5JBItFr7Bey7SOAc0IjJGJtRkZo2FA5ObGlPEwHOEBHYB88DR7D2MR4GEkcphWppVOUsRf2SJHJKlw9GAcfXnP4DLbLPHMoIXbrsAOtd39Y/OFqevESjGnOe+IzjTm8mBZlkIdJqH/9E37Uunp1ZtV+DSehm89mJPkhfF0Ks4Gn51m+NRpcf6eFH4wrBGmugwpdz4JopjlYnEdxEHgpfWLj52mtSZxBXZYbTohtmXlhRe0L31JS56yLrXKMXnjuYN9kCKwgv2v4I8XppaN9QLL+mOnjR8rQZ7XAwSWzmQFbw4pTlwTdZt2hFKfCTlVUxwu6e5GZg4cHbiw2DMkBm5B+RvBlJc30ptd03Mb1ICbffc/Pvnop3h5Y4b0VNhLchd45kmuHGCksOAQsPaXnAg22JFf4S42MIDmEphw8H+JMrpE6HKm1agmVSYmCECy6JNVwhbmuRVu8kKleGlhd294EJaC0C7pa7oBuy05gc5K2vErShqsk32npCqYZgXWSREkbVGXC/Oq5Pn8iscEBT90PJ6SkNoT+qDdBvEBGHxpdT4tustIL2eVglnheYqQluMTgr9Q1xTYGXf5Ts7VfapRgCSVWcaRx8VfDpVtNmA1SZPMKok2t5pr2+Y6XgZFv6LFQ8icq05ZfUccTocYQ1Fc6ESxdbrtvfXuXrvREa9nQqwko67IcqdUZxl3MUEtdrtoWlvock4jAfnCGW4e+cL56mSzpUsRrys6b4EuMSMOz8wER1JyjkknnmcSZFOuFhMORSx4tcRugOVCqRxoJ/yVEpIXgEWleCTqAnMvL/T5qyFYYJ2euiK5SHICKv+6SjdduTB7jq2b95pEg20+BDHh5oMwo24+EFp28yHQ1lNCJAIxAO7iwKwRsXqOzcMhzPg48M5BPAbosfsiuK6pGEMLXb7giFvT5DKcKdaj68HhPDiyUCRk3EHj3O2OWDVP17a41mv0tFwaeVA1WnTVZYfNyPhNWnlUT/sm+SMS3cY4hpgpFmMZSFLe94dtWkDN3ZqR3JCz1zQTvzN/ZiNpXHQrVW4CE/NlSW0N4DlKnhw1HF7AMyVhRM7BoGuEHiQtajWael7xW6qbkC1FIMQ29WFZxs4kuV8pYY7Z2Omst9pbzatmE752G7sdBfn9QhYudC1MAd/V9OLVMUxQ1ADnWGgFxlkR7DmXy6hv78g0sMiH3KUz/M28ImTRJTSPcxlNVn+hiLEk19LchQUa108Kuz/VlXgNxbXB8g6gDr6oWZLHt+/4ZB+0M/Wo1BHmQZRXE5zFpsCj+OsQi0SvMzaevfNLGn3SRCLyZ8uyjT9V7crLZhR2WU19VOzCiaiJT8381irGf4vnTNCmCkaVZdrnTXtv/obGIhs8tqpwCaD6Dmg1A0ktQ5H6V+t7FjhcVaxUunufq+4zZxBVpMyZQg2cv0KPkWJ2Gzu7mMAGQ9tcL0Twgr6IxsU/yiDapXMv+VYN/FLRyNOrDtkfBYFhiH2RPIL0aKWvvi6nkP+PtVdItP8f2ArEYc7bCiTIz3crSI6GSPq/sM/PoP97CxjE/UItOCeguSALcxeQN8gHXq9nRfhMinDxhOdNbHI77MiC3cDWF2pHTW5shmsduhMpxPNlDtYsVIlcmABQyp1HxoW72+p0YbbWG52Oud5qdreaew2z1TQb7XarXXDSeElUmehsJqm7jNzkvEAG8v8m/UU+0uCIKrSxzl8SM7BIrTS7IIb/20+5g7zl3NFh8R7Rc6pWyZh6ahRJeICRwRN1ib+1hiRV+eudxZ+FHrYKhqWi3TDR6EuP4c0qgOfgm2sV8Kjya6/gLV1huTtx0gn+H7oKh87RY9x4w0I82cITV3Ga+aeX2qAlze60wb9ZIdxro7rOhnud16xMeJjQLCQsn0kgFNmt9AA15p6yMeX1Nh1/kgXTdVxXLKsX8ZohvXn/qyNpd2IYisWcIPaCIywwjMyBg3+5Ro2iSLA5DGN84zv7VuzecorwyfcGJXUWVP9MUagUwVQqIvS1KpaaNSwOiWYwC/+BNd4d4ELMHJmcGiXWu6Kh6i8WCBgUd1cthoqcV56PjF6clKHDfJcYGi78y4DYpTJQLfQnYct6SuPw6eFGinZsHTomJtZBSEpLfFRILMiGBV7LkJA9itvs9Bcxh5jtfUFAi5w5QaB7iXij3zONxq65Rq4uJtWmwCmsOaflw0V/G3Ghs2vFcgddA/ZGt5GdMDa3mt1Ge6exsSUdO/mczr4tkCQQLynYxPWFRSOrejmTaD4VmIn5nAqx3urKMvwn5BB1ijuTk1Xq9XFSowCHKc9IgR8ZK1vPNvGvFJgpQVztA16EkJ60RLnNyDaKMGBP52NAEoBpxemopf8CQHxXDQ==')).decode('utf-8'))


## 输出
最终 CSV 包含序列、ipTM、pTM、HADDOCK3 emscoring score、PRODIGY 预测结合自由能及方向说明。HADDOCK3 分数越低越好；PRODIGY kcal/mol 越低、越负表示预测结合更强。ZIP 同时包含 HADDOCK3 能量最小化 PDB。
